In [1]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr

import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import numpy as np
import time

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [10]:

#WR2 data
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']


data_paths =[f'wr2/bar-n{j}-{k}-3.csv'for j in sec_names for k in third_names]#TODO add data
dist_paths = [f'wr2/bar-n{j}-{k}-dist.csv'for j in sec_names for k in third_names]#TODO add data
num_robots = [5*i for i in range(2,3)]

method = 5
file_type = 'csv'
is_depot_task = False


In [3]:

#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [13]:
num_robots = np.random.randint(1,[2,3,4,5,6,7,8,9,10,11],[10,10])
num_robots

array([[ 1,  2,  2,  2,  5,  1,  6,  4,  2,  4],
       [ 1,  1,  1,  1,  2,  4,  3,  2,  4,  8],
       [ 1,  2,  1,  4,  1,  3,  7,  5,  2,  7],
       [ 1,  1,  3,  1,  2,  2,  5,  8,  2,  8],
       [ 1,  1,  2,  3,  2,  6,  6,  2,  2,  8],
       [ 1,  2,  2,  1,  5,  5,  1,  7,  9,  7],
       [ 1,  2,  1,  1,  2,  3,  4,  7,  1,  2],
       [ 1,  2,  1,  1,  3,  5,  6,  5,  7, 10],
       [ 1,  1,  2,  2,  2,  5,  2,  6,  9, 10],
       [ 1,  2,  3,  1,  4,  4,  7,  1,  7,  3]])

In [14]:
num_robots_fixed = np.array([[ 1,  2,  3,  1,  1,  2,  3,  3,  2,  9],
       [ 1,  1,  1,  1,  1,  1,  3,  4,  8,  8],
       [ 1,  2,  2,  1,  2,  3,  5,  4,  4, 10],
       [ 1,  1,  3,  4,  1,  2,  7,  1,  6,  6],
       [ 1,  1,  1,  3,  5,  2,  3,  6,  7,  7],
       [ 1,  1,  3,  2,  1,  5,  5,  4,  4,  9],
       [ 1,  2,  3,  2,  4,  1,  4,  7,  7,  6],
       [ 1,  1,  3,  4,  3,  6,  3,  2,  8,  4],
       [ 1,  2,  1,  1,  5,  3,  3,  4,  4,  9],
       [ 1,  2,  2,  3,  4,  4,  7,  5,  9,  7]])

num_robots_fixed2 = np.array([[1, 1, 2, 1, 5, 1, 7, 7, 4, 3],
       [1, 2, 1, 3, 5, 3, 3, 2, 4, 4],
       [1, 1, 2, 1, 4, 1, 4, 4, 1, 7],
       [1, 2, 3, 3, 2, 5, 5, 8, 9, 8],
       [1, 1, 3, 4, 4, 6, 6, 3, 6, 4],
       [1, 1, 1, 4, 1, 2, 5, 2, 5, 5],
       [1, 1, 2, 1, 5, 6, 1, 6, 4, 1],
       [1, 1, 1, 4, 2, 2, 3, 5, 6, 5],
       [1, 2, 1, 4, 4, 2, 2, 4, 4, 6],
       [1, 2, 1, 3, 1, 2, 5, 8, 9, 4]])

In [4]:
# CPLEX data generated
# data_idcs = [4,5,16,18]
# data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [10]
file_type = 'csv'
i = 0
is_depot_task = False
for data_path in data_paths:
    for num_robot in num_robots:
        for i in range(10):
            try:
                # cur_num_robots = []
                # cur_num_robots = num_robots_fixed2[:,i]
        

                raw = get_data(data_path, file_type, is_depot_task)
                avg_num_robots = np.mean([raw.tasks[i].n for i in range(len(raw.tasks))])
                print(f'Solving for {data_path} with {num_robot} robots with average n_j = {avg_num_robots}')
                method = 5
                processed_data_vrp = process(raw, method)
                solve_result_vrp = solve_pdpcr(processed_data_vrp, num_robot, method)
                # method = 2
                # processed_data_flow_old = process(raw, method)
                # solve_result_flow_old = solve_pdpcr(processed_data_flow_old, num_robot, method)
                

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)

                # solve_log.append(solve_result_flow_old['solve_time'])
                # solve_log.append(solve_result_flow_old['sol'].objective_value)
                # solve_log.append(solve_result_flow_old['processed_nodes'])
                # solve_log.append(solve_result_flow_old['gap'])

                solve_log.append(solve_result_vrp['solve_time'])
                solve_log.append(solve_result_vrp['sol'].objective_value)
                solve_log.append(solve_result_vrp['processed_nodes'])
                solve_log.append(solve_result_vrp['gap'])

                now = datetime.datetime.now()
                solve_log.append(avg_num_robots)
                solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
                with open('solve_results/solve_log_li_lim_vrp.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)
            i += 1

Solving for lim/200/LC1_2_1-5-uniform.csv with 10 robots with average n_j = 3.018867924528302
C:\Users\CTNGUY~1\AppData\Local\Temp\PDPCR_Flow_Based_Makespan.lp
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               1800
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 3851 rows and 215 columns.
MIP Presolve modified 33290 coefficients.
Aggregator did 3530 substitutions.
Reduced MIP has 27502 rows, 19475 columns, and 88708 nonzeros.
Reduced MIP has 11448 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.13 sec. (103.93 ticks)
Probing time = 0.09 sec. (10.97 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 27502 rows, 19475 columns, and 88708 nonzeros.
Reduced MIP has 11448 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.08 sec. (55.09 ticks)
Probing time = 0.13 sec. (10.95 ticks)
Cl

In [ ]:
# read file 'solve_results/solve_log_generated_progressive_large_nj.csv' to pandas dataframe, it has the following columns:
# data_name, total_robots, solve_time_vrp, obj_value_vrp, processed_nodes_vrp, gap_vrp, solve_time_flow, obj_value_flow, processed_nodes_flow, gap_flow, avg_num_robots,date
import pandas as pd
df_large_nj = pd.read_csv('solve_results/solve_log_generated_progressive_large_nj.csv', header=None)
df_large_nj.columns = ['data_name', 'total_robots', 'solve_time_vrp', 'obj_value_vrp', 'processed_nodes_vrp', 'gap_vrp', 'solve_time_flow', 'obj_value_flow', 'processed_nodes_flow', 'gap_flow','avg_num_robots', 'date']


In [5]:
first_names = ['lc','lr','lrc']
sec_names = [i for i in range(101,115)] +[i for i in range(201,215)]
# third_names = [i for i in range(1,11)]
data_paths =[f'lim/100/{i}{j}-5-uniform.csv'for i in first_names for j in sec_names]#
num_robots = [10*i for i in range(1,2)]
file_type = 'csv'
is_depot_task = False

In [8]:
#we have data_path = 'lim/100/lc101-5-uniform.csv', get data_name = 'lc101-5-uniform'
data_names = [data_path.split('/')[2].split('.')[0] for data_path in data_paths]

In [7]:
## Mass solve using tabu_search
num_robots = [10,20,30]
for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/1.3)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/1.3)
            cfg.TABU_NUM_REROUTE_NEIGHBORS = int(len(tasks)/1.2)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = int(cfg.TABU_NUM_REROUTE_NEIGHBORS/3)
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 30
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 7
            cfg.TABU_MAX_ITER = 5000
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            tik1 = time.time()
            tabu = heuristic.TabuSearch(processed_h,c_travel, c_operate)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

20-Jun-24 13:34:39:INFO:Solving data lim/200/LC1_2_1-5-uniform.csv with 10 robots and 106 tasks
20-Jun-24 13:34:39:INFO:TABU SEARCH PARAMETERS
20-Jun-24 13:34:39:INFO:RANDOM_SEED: 15
20-Jun-24 13:34:39:INFO:NUM_REVERSE_NEIGHBORS: 29
20-Jun-24 13:34:39:INFO:NUM_REROUTE_NEIGHBORS: 88
20-Jun-24 13:34:39:INFO:MAX_TL_ADD_SIZE: 81
20-Jun-24 13:34:39:INFO:MAX_TL_DROP_SIZE: 81
20-Jun-24 13:34:39:INFO:MAX_ITER: 5000
20-Jun-24 13:34:39:INFO:FIND_FROM_ELITE_AFTER: 500
20-Jun-24 13:34:39:INFO:RESTART_AFTER: 1000
20-Jun-24 13:34:39:INFO:REROUTE_ON_CRITICAL: True
20-Jun-24 13:34:39:INFO:REVERSE_ON_CRITICAL: True
20-Jun-24 13:34:39:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True


Viet generalized sol is 952.2
Viet generalized sol is 934.56
Viet generalized sol is 986.27
Viet generalized sol is 1071.13
Viet generalized sol is 1015.71
Viet generalized sol is 955.81
Viet generalized sol is 1024.79
Viet generalized sol is 962.29
Viet generalized sol is 974.33
Viet generalized sol is 976.27
Viet generalized sol is 921.52
Viet generalized sol is 1039.78
Viet generalized sol is 978.29
Viet generalized sol is 969.89
Viet generalized sol is 986.26
Viet generalized sol is 980.14
Viet generalized sol is 1140.13
Viet generalized sol is 1001.57
Viet generalized sol is 986.09
Viet generalized sol is 1044.86
Viet generalized sol is 929.07
Viet generalized sol is 982.78
Viet generalized sol is 955.33
Viet generalized sol is 953.54
Viet generalized sol is 1024.34
Viet generalized sol is 933.63
Viet generalized sol is 1043.59
Viet generalized sol is 958.66
Viet generalized sol is 943.5


20-Jun-24 13:34:43:INFO:Initial makespan: 921.5219918762432


Viet generalized sol is 1087.69


20-Jun-24 13:34:43:INFO:Best fitness updated: 918.4563032030701
20-Jun-24 13:34:43:INFO:Iteration 0 done
20-Jun-24 13:34:43:INFO:Best fitness updated: 916.283116887204
20-Jun-24 13:34:44:INFO:Best fitness updated: 914.1722933166994
20-Jun-24 13:34:44:INFO:Best fitness updated: 912.7787571595452
20-Jun-24 13:34:44:INFO:Best fitness updated: 911.8879481732065
20-Jun-24 13:34:44:INFO:Best fitness updated: 911.2447986638347
20-Jun-24 13:34:49:INFO:Best fitness updated: 907.4553723949516
20-Jun-24 13:34:51:INFO:Best fitness updated: 907.1484958136102
20-Jun-24 13:34:51:INFO:Best fitness updated: 905.8188117133593
20-Jun-24 13:34:54:INFO:Iteration 50 done
20-Jun-24 13:35:04:INFO:Iteration 100 done
20-Jun-24 13:35:15:INFO:Iteration 150 done
20-Jun-24 13:35:28:INFO:Iteration 200 done
20-Jun-24 13:35:41:INFO:Iteration 250 done
20-Jun-24 13:35:59:INFO:Iteration 300 done
20-Jun-24 13:36:16:INFO:Iteration 350 done
20-Jun-24 13:36:34:INFO:Iteration 400 done
20-Jun-24 13:36:51:INFO:Iteration 450 don

Viet generalized sol is 916.5
Viet generalized sol is 947.02
Viet generalized sol is 934.66
Viet generalized sol is 941.91
Viet generalized sol is 922.08
Viet generalized sol is 998.19
Viet generalized sol is 956.46
Viet generalized sol is 928.73
Viet generalized sol is 928.22
Viet generalized sol is 934.09
Viet generalized sol is 970.14
Viet generalized sol is 949.03
Viet generalized sol is 926.36
Viet generalized sol is 933.63
Viet generalized sol is 929.03
Viet generalized sol is 939.56
Viet generalized sol is 971.77
Viet generalized sol is 952.33
Viet generalized sol is 922.62
Viet generalized sol is 958.92
Viet generalized sol is 1005.78
Viet generalized sol is 952.56
Viet generalized sol is 934.77
Viet generalized sol is 928.73
Viet generalized sol is 943.9
Viet generalized sol is 938.91
Viet generalized sol is 936.96
Viet generalized sol is 954.22
Viet generalized sol is 941.91
Viet generalized sol is 931.34


... (332 outputs left out)

21-Jun-24 15:18:01:INFO:Initial makespan: 4215.173313934047


Viet generalized sol is 4457.44


21-Jun-24 15:18:02:INFO:Best fitness updated: 4201.235465464981
21-Jun-24 15:18:02:INFO:Iteration 0 done
21-Jun-24 15:18:04:INFO:Best fitness updated: 4190.268875258663
21-Jun-24 15:18:05:INFO:Best fitness updated: 4170.010173863606
21-Jun-24 15:18:07:INFO:Best fitness updated: 4163.028625128662
21-Jun-24 15:18:08:INFO:Best fitness updated: 4161.34235983058
21-Jun-24 15:18:09:INFO:Best fitness updated: 4160.498638337876
21-Jun-24 15:18:11:INFO:Best fitness updated: 4160.08576571969
21-Jun-24 15:18:12:INFO:Best fitness updated: 4157.287229172498
21-Jun-24 15:18:15:INFO:Best fitness updated: 4154.135514145493
21-Jun-24 15:18:17:INFO:Best fitness updated: 4143.490734762569
21-Jun-24 15:18:18:INFO:Best fitness updated: 4134.747007328659
21-Jun-24 15:18:19:INFO:Best fitness updated: 4132.335246440545
21-Jun-24 15:18:20:INFO:Best fitness updated: 4130.003630194856
21-Jun-24 15:18:22:INFO:Best fitness updated: 4121.978266021317
21-Jun-24 15:18:41:INFO:Best fitness updated: 4119.610666366137
2

KeyboardInterrupt: 

In [ ]:
## mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve Viet 

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
consider_process_time =True
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_viet = heuristic.VietSolverOrig(processed_h, c_travel, c_operate)
            solve_viet.solve(num_robot,consider_process_time)
            tok = time.time()
            time_viet = tok - tik
            print(f'makespan: {solve_viet.get_makespan()}')
            print('Done')

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_viet.get_makespan().round(2))
            solve_log.append(np.round(time_viet,2))
            with open('solve_log_h_Viet_09-04-process-time.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)


In [ ]:
## Mass solve Viet k tasks

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

tasks_each_iters = [5]
consider_process_time =False
for task_each_ite in tasks_each_iters:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)
                tik = time.time()
                solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                sol = solve_viet.solve(num_robot,task_each_ite,consider_process_time)
                tok = time.time()
                time_viet = tok - tik
                makespan = sol.get_makespan()
                print(f'makespan: {makespan}')
                print('Done')

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(task_each_ite)
                solve_log.append(makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                with open('solve_log_h_Viet k task large instances 23-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
a ={0: 0.0, 1: 599.9817763899168, 2: 271.90904527065663, 3: 283.1551518161024, 4: 142.61641735789044, 5: 239.53321272842314, 6: 503.8917411508151, 7: 320.3896980241406, 8: 347.00099826945745, 9: 177.80785134520917, 10: 194.39276118209756, -1: 0}

In [ ]:
for i in a:
    print(i)

0
1
2
3
4
5
6
7
8
9
10
-1


In [ ]:
## Mass solve Viet k tasks with random k

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

ite = 100
tasks_each_iter_ranges = [[1,3],[1,4],[1,5]]
consider_process_time =False
for task_each_ite in tasks_each_iter_ranges:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)

                solve_result_h_jp = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
                TT_makespan = solve_result_h_jp.get_makespan()

                tik = time.time()
                cur_best = np.infty
                cur_worst = -np.infty
                makespan_results = []
                no_better_TT = 0

                for i in range(ite):
                    solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                    sol = solve_viet.solve(num_robot,'random',consider_process_time,task_range=task_each_ite)
                    makespan = sol.get_makespan()
                    makespan_results.append(makespan)
                    if makespan < cur_best:
                        cur_best = makespan
                    if makespan > cur_worst:
                        cur_worst = makespan
                    if makespan <= TT_makespan:
                        no_better_TT +=1
                avg_makespan = np.mean(makespan_results)
                tok = time.time()
                time_viet = tok - tik
            
                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(str(f'{task_each_ite[0]}-{task_each_ite[1]}'))
                solve_log.append(cur_best.round(2))
                solve_log.append(cur_worst.round(2))
                solve_log.append(avg_makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                solve_log.append(no_better_TT)
                with open('solve_log_h_Viet k task random large instances 29-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [ ]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [ ]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [ ]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [ ]:
merged_df.to_latex('merged_df.tex', index=True)

In [ ]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [ ]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [ ]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [ ]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)